In [1]:
#Load required packages
library(tidyverse)
library(geosphere)

Warning message:
“package ‘tidyr’ was built under R version 4.4.3”
Warning message:
“package ‘readr’ was built under R version 4.4.3”
Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘stringr’ was built under R version 4.4.3”
Warning message:
“package ‘forcats’ was built under R version 4.4.3”
Warning message:
“package ‘lubridate’ was built under R version 4.4.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
#Get list of BAMfiles and check length
bamlist<-read.table('/home/jbos/Moz_reads/bam_names_grp137.txt')
nrow(bamlist)

[1] 236

In [3]:
#Little function to get sample names from BAM filepaths
bamnames<-function(bam){
    a<-strsplit(bam,'files/')[[1]][2]
    b<-strsplit(a,'.s')[[1]][1]
    return(b)
    }

In [4]:
indlist<-apply(bamlist,FUN=bamnames,MARGIN=1)

In [6]:
#Read in NGSrelate output
rels<-read.table('/scratch/jbos/Moz_aligned_mil/glf_grp137/newres',header=TRUE)
colnames(rels)
nrow(rels)

[1] "a"                      "b"                      "nSites"                
 [4] "J9"                     "J8"                     "J7"                    
 [7] "J6"                     "J5"                     "J4"                    
[10] "J3"                     "J2"                     "J1"                    
[13] "rab"                    "Fa"                     "Fb"                    
[16] "theta"                  "inbred_relatedness_1_2" "inbred_relatedness_2_1"
[19] "fraternity"             "identity"               "zygosity"              
[22] "X2of3_IDB"              "FDiff"                  "loglh"                 
[25] "nIter"                  "bestoptimll"            "coverage"              
[28] "X2dsfs"                 "R0"                     "R1"                    
[31] "KING"                   "X2dsfs_loglike"         "X2dsfsf_niter"

[1] 27730

In [7]:
#Do some shenanigans to properly name all individuals in each relative pair
indlist_a<-as.data.frame(indlist)
indlist_a$a<-c(seq(0,235,1))
colnames(indlist_a)<-c('IndA','a')

In [8]:
relsa<-left_join(rels,indlist_a)

Joining with `by = join_by(a)`


In [9]:
indlist_b<-as.data.frame(indlist)
indlist_b$b<-c(seq(0,235,1))
colnames(indlist_b)<-c('IndB','b')

In [10]:
rels_all<-left_join(relsa,indlist_b)

Joining with `by = join_by(b)`


In [12]:
#Check median KING relatedness
median(rels_all$KING)

[1] 0.1283985

In [13]:
#Get list of DB individuals
grp1<-read.table('/home/jbos/Moz_reads/bam_names_grp1.txt')
grp1<-sapply(strsplit(grp1$V1, "files/"), "[", 2)
grp1<-sapply(strsplit(grp1, ".sor"), "[", 1)

In [14]:
#Sort out DB individuals
rels_all$BayA<-0
rels_all$BayA[rels_all$IndA %in% grp1]<-1

rels_all$BayB<-0
rels_all$BayB[rels_all$IndB %in% grp1]<-1

In [18]:
table(rels_all$BayA)


    0     1 
13203 14527 

In [16]:
DB<-rels_all[rels_all$BayA==1,]
DB<-DB[DB$BayB==1,]

In [17]:
#Check median relatedness of DB vs all, just for fun
median(rels_all$KING)
median(DB$KING)

[1] 0.1283985

[1] 0.160522

In [18]:
#This output has too many columns
which(colnames(rels_all)=='KING')

[1] 31

In [19]:
#Look at all KING coefficients in order
rels_all[order(-rels_all$KING),31]

[1]  0.440890  0.435617  0.238824  0.238658  0.227789  0.226161  0.221046
    [8]  0.216856  0.212239  0.211282  0.209890  0.208177  0.207401  0.203856
   [15]  0.198747  0.194990  0.193659  0.192583  0.192014  0.191867  0.191629
   [22]  0.191571  0.191509  0.190703  0.190577  0.190056  0.189347  0.189222
   [29]  0.189065  0.188384  0.188358  0.188277  0.188028  0.187894  0.187630
   [36]  0.187588  0.187027  0.186651  0.186596  0.186489  0.186426  0.186391
   [43]  0.186246  0.186229  0.186153  0.186051  0.186026  0.186016  0.185933
   [50]  0.185873  0.185861  0.185776  0.185742  0.185666  0.185542  0.185501
   [57]  0.185320  0.185129  0.185115  0.184844  0.184790  0.184739  0.184708
   [64]  0.184473  0.184380  0.184090  0.183928  0.183895  0.183867  0.183821
   [71]  0.183771  0.183741  0.183551  0.183523  0.183514  0.183379  0.183277
   [78]  0.183260  0.183217  0.183088  0.182975  0.182972  0.182908  0.182838
   [85]  0.182788  0.182768  0.182743  0.182684  0.182619  0.182389  0.182322
   [92]  0.182138  0.182133  0.182046  0.182043  0.182013  0.181963  0.181761
   [99]  0.181741  0.181620  0.181549  0.181548  0.181517  0.181509  0.181488
  [106]  0.181268  0.181222  0.181171  0.181149  0.181130  0.181024  0.180995
  [113]  0.180966  0.180943  0.180883  0.180866  0.180811  0.180663  0.180657
  [120]  0.180556  0.180554  0.180540  0.180427  0.180427  0.180424  0.180307
  [127]  0.180275  0.180269  0.180249  0.180226  0.180195  0.180190  0.180177
  [134]  0.180134  0.180027  0.180018  0.179998  0.179931  0.179915  0.179735
  [141]  0.179619  0.179616  0.179603  0.179576  0.179531  0.179408  0.179384
  [148]  0.179374  0.179365  0.179304  0.179249  0.179245  0.179222  0.179201
  [155]  0.179025  0.178968  0.178965  0.178927  0.178803  0.178702  0.178688
  [162]  0.178669  0.178637  0.178637  0.178477  0.178456  0.178331  0.178307
  [169]  0.178266  0.178150  0.178069  0.177858  0.177837  0.177823  0.177771
  [176]  0.177707  0.177608  0.177603  0.177601  0.177600  0.177557  0.177546
  [183]  0.177505  0.177485  0.177483  0.177352  0.177333  0.177294  0.177252
  [190]  0.177089  0.177036  0.177014  0.177003  0.176956  0.176856  0.176850
  [197]  0.176809  0.176808  0.176781  0.176752  0.176636  0.176592  0.176591
  [204]  0.176588  0.176557  0.176546  0.176492  0.176421  0.176373  0.176353
  [211]  0.176322  0.176203  0.176143  0.176139  0.176115  0.176056  0.176005
  [218]  0.176005  0.175942  0.175914  0.175888  0.175840  0.175777  0.175776
  [225]  0.175753  0.175682  0.175653  0.175620  0.175554  0.175552  0.175543
  [232]  0.175502  0.175478  0.175462  0.175428  0.175363  0.175358  0.175353
  [239]  0.175278  0.175246  0.175213  0.175212  0.175095  0.174977  0.174962
  [246]  0.174945  0.174860  0.174783  0.174755  0.174726  0.174527  0.174443
  [253]  0.174424  0.174405  0.174294  0.174282  0.174243  0.174241  0.174228
  [260]  0.174225  0.174216  0.174209  0.174206  0.174204  0.174147  0.174124
  [267]  0.174112  0.174095  0.174042  0.174032  0.174003  0.173972  0.173971
  [274]  0.173941  0.173932  0.173923  0.173904  0.173895  0.173883  0.173883
  [281]  0.173824  0.173823  0.173804  0.173795  0.173763  0.173706  0.173699
  [288]  0.173684  0.173668  0.173654  0.173617  0.173532  0.173521  0.173494
  [295]  0.173463  0.173459  0.173454  0.173417  0.173408  0.173403  0.173374
  [302]  0.173369  0.173355  0.173354  0.173331  0.173310  0.173303  0.173283
  [309]  0.173242  0.173241  0.173202  0.173179  0.173177  0.173135  0.173131
  [316]  0.173126  0.173104  0.173053  0.173053  0.173049  0.173004  0.173003
  [323]  0.172992  0.172975  0.172911  0.172898  0.172870  0.172861  0.172852
  [330]  0.172834  0.172829  0.172817  0.172814  0.172809  0.172796  0.172770
  [337]  0.172769  0.172761  0.172737  0.172716  0.172704  0.172684  0.172672
  [344]  0.172667  0.172652  0.172632  0.172622  0.172612  0.172580  0.172575
  [351]  0.172568  0.172559  0.172538  0.172526  0.172510  0.172491  0.172489
  [358]  0.172486  0.17247

In [20]:
#Sort out potential clones
clones<-rels_all[rels_all$KING>0.35,]

In [22]:
#Looks like two potential clone pairs
clones

,a,b,nSites,J9,J8,J7,J6,J5,J4,J3,⋯,X2dsfs,R0,R1,KING,X2dsfs_loglike,X2dsfsf_niter,IndA,IndB,BayA,BayB
,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<chr>,<dbl>,<dbl>
25961,176,177,3984467,0.192783,5.0e-05,0.807166,0e+00,0,0,0,⋯,"6.355650e-01,3.583657e-02,1.357185e-10,4.053355e-02,2.613623e-01,4.224555e-04,3.476200e-10,4.644084e-04,2.581565e-02",0,3.383025,0.435617,-4363988,11,ACR_416,ACR_419,0,0
26786,192,194,3986482,0.177092,5.3e-05,0.822854,1e-06,0,0,0,⋯,"6.400557e-01,3.466675e-02,1.511623e-09,3.511383e-02,2.635376e-01,4.343429e-04,1.845300e-09,4.501945e-04,2.574157e-02",0,3.729388,0.440890,-4273474,10,ACR_502,ACR_505,0,0


In [23]:
#Figure out which individuals are clones
clones$IndA
clones$IndB

[1] "ACR_416" "ACR_502"

[1] "ACR_419" "ACR_505"

In [26]:
metadat<-read.csv('/home/jbos/Moz_reads/Acropora_moz_metadat_certainty.csv')

In [27]:
leading_zeros<-function(num){
    if (nchar(as.character(num))<2){
        return(paste0('00',num))
        } else {
        if (nchar(as.character(num))<3){
            return(paste0('0',num))
            } else {
       return(as.character(num))
            }
        }
    }

In [28]:
#Add leading zero where necessary
metadat$Numero_do_tubo<-sapply(metadat$Numero_do_tubo,leading_zeros)

In [29]:
metadat$IND<-paste("ACR_",metadat$Numero_do_tubo,sep="")

In [31]:
clones1<-metadat[metadat$IND %in% c('ACR_416','ACR_419'),]
clones2<-metadat[metadat$IND %in% c('ACR_502','ACR_505'),]

In [32]:
clones1

,Numero_do_tubo,Latitude,Longitude,Profundidade_metros,Dia,Mes,Ano,Bleach,SppIDCertainty,Loc,Spp,IND
,<chr>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
162,416,-12.96453,40.54864,5.4,13,5,2024,N,C,W,A,ACR_416
167,419,-12.96453,40.54864,4.2,13,5,2024,N,C,W,A,ACR_419


In [38]:
clones2
p1<-c(clones2$Longitude[1],clones2$Latitude[1])
p2<-c(clones2$Longitude[2],clones2$Latitude[2])

,Numero_do_tubo,Latitude,Longitude,Profundidade_metros,Dia,Mes,Ano,Bleach,SppIDCertainty,Loc,Spp,IND
,<chr>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
181,502,-12.96111,40.55250,7.8,21,5,2024,N,C,W,A,ACR_502
202,505,-12.96556,40.54583,4.2,23,5,2024,N,C,W,A,ACR_505


In [39]:
distHaversine(p1,p2)

[1] 876.3193

In [25]:
#All at Wimbe, incidentally
'ACR_416' %in% grp1
'ACR_502' %in% grp1
'ACR_419' %in% grp1
'ACR_505' %in% grp1

[1] FALSE

[1] FALSE

[1] FALSE

[1] FALSE

In [46]:
table(rels_all$KING>0.177)


FALSE  TRUE 
27537   193 

In [47]:
mayberels<-rels_all[rels_all$KING>0.177,]

In [48]:
mayberels[order(-mayberels$KING),c('IndA','IndB','KING','X2dsfs_loglike')]

,IndA,IndB,KING,X2dsfs_loglike
,<chr>,<chr>,<dbl>,<dbl>
26786,ACR_502,ACR_505,0.440890,-4273474
25961,ACR_416,ACR_419,0.435617,-4363988
751,ACR_019,ACR_702,0.238824,-5365158
4093,ACR_481,ACR_541,0.238658,-5607580
4324,ACR_486,ACR_598,0.227789,-5587177
11812,ACR_707,ACR_729,0.226161,-5581459
15650,ACR_304,ACR_314,0.221046,-2805856
23055,ACR_568,ACR_558,0.216856,-5605434
6823,ACR_535,ACR_541,0.212239,-5659277


In [49]:
which(colnames(rels_all)=='theta')

[1] 16

In [50]:
rels_all[order(-rels_all$theta),16]

[1] 0.411440 0.403595 0.186726 0.185896 0.161213 0.151387 0.146369 0.145220
    [9] 0.142596 0.118941 0.115048 0.112129 0.110983 0.104490 0.103316 0.103243
   [17] 0.094674 0.089631 0.088784 0.083895 0.082448 0.081139 0.080835 0.078658
   [25] 0.078429 0.077988 0.077146 0.076594 0.076478 0.076341 0.075925 0.075502
   [33] 0.073505 0.073313 0.073267 0.072982 0.072849 0.072354 0.072343 0.072000
   [41] 0.071964 0.071300 0.071203 0.071103 0.071085 0.070914 0.070881 0.070500
   [49] 0.070455 0.070094 0.070045 0.069968 0.069945 0.069932 0.069791 0.069789
   [57] 0.069616 0.069488 0.069465 0.069436 0.069431 0.069410 0.069356 0.069276
   [65] 0.069262 0.069256 0.069250 0.069196 0.069129 0.069097 0.069081 0.068940
   [73] 0.068909 0.068885 0.068869 0.068781 0.068749 0.068609 0.068557 0.068551
   [81] 0.068463 0.068462 0.068449 0.068393 0.068362 0.068348 0.068223 0.068107
   [89] 0.068093 0.068085 0.067968 0.067817 0.067746 0.067708 0.067688 0.067601
   [97] 0.067568 0.067493 0.067350 0.067314 0.067257 0.067233 0.067218 0.067211
  [105] 0.067168 0.067081 0.067037 0.066977 0.066965 0.066956 0.066918 0.066918
  [113] 0.066842 0.066832 0.066770 0.066764 0.066760 0.066758 0.066749 0.066740
  [121] 0.066730 0.066711 0.066707 0.066679 0.066661 0.066658 0.066650 0.066612
  [129] 0.066595 0.066583 0.066569 0.066555 0.066461 0.066438 0.066423 0.066422
  [137] 0.066306 0.066251 0.066187 0.066174 0.066167 0.066109 0.066091 0.066071
  [145] 0.066067 0.066032 0.066003 0.065994 0.065974 0.065970 0.065941 0.065915
  [153] 0.065889 0.065843 0.065841 0.065840 0.065755 0.065745 0.065743 0.065714
  [161] 0.065676 0.065667 0.065641 0.065621 0.065608 0.065524 0.065476 0.065405
  [169] 0.065385 0.065339 0.065331 0.065281 0.065273 0.065272 0.065270 0.065264
  [177] 0.065229 0.065220 0.065215 0.065208 0.065164 0.065154 0.065123 0.065122
  [185] 0.064980 0.064960 0.064948 0.064889 0.064886 0.064865 0.064809 0.064781
  [193] 0.064776 0.064762 0.064745 0.064744 0.064708 0.064693 0.064681 0.064647
  [201] 0.064640 0.064606 0.064602 0.064599 0.064595 0.064559 0.064540 0.064529
  [209] 0.064472 0.064458 0.064425 0.064398 0.064394 0.064359 0.064354 0.064338
  [217] 0.064304 0.064284 0.064236 0.064228 0.064227 0.064220 0.064202 0.064179
  [225] 0.064179 0.064156 0.064142 0.064136 0.064128 0.064120 0.064119 0.064064
  [233] 0.064036 0.064017 0.064016 0.064006 0.063987 0.063946 0.063891 0.063863
  [241] 0.063849 0.063832 0.063829 0.063825 0.063819 0.063797 0.063757 0.063754
  [249] 0.063750 0.063745 0.063725 0.063724 0.063715 0.063700 0.063698 0.063631
  [257] 0.063601 0.063595 0.063573 0.063564 0.063540 0.063532 0.063511 0.063499
  [265] 0.063490 0.063489 0.063488 0.063465 0.063440 0.063406 0.063389 0.063373
  [273] 0.063364 0.063337 0.063313 0.063287 0.063284 0.063278 0.063275 0.063262
  [281] 0.063251 0.063236 0.063208 0.063165 0.063148 0.063142 0.063140 0.063115
  [289] 0.063105 0.063098 0.063093 0.063091 0.063056 0.063052 0.063014 0.063005
  [297] 0.062992 0.062988 0.062980 0.062978 0.062968 0.062961 0.062957 0.062939
  [305] 0.062936 0.062893 0.062888 0.062874 0.062869 0.062868 0.062850 0.062796
  [313] 0.062792 0.062786 0.062783 0.062773 0.062766 0.062764 0.062762 0.062755
  [321] 0.062718 0.062718 0.062717 0.062686 0.062684 0.062678 0.062672 0.062651
  [329] 0.062650 0.062645 0.062640 0.062629 0.062623 0.062609 0.062585 0.062559
  [337] 0.062555 0.062540 0.062537 0.062483 0.062457 0.062434 0.062407 0.062400
  [345] 0.062397 0.062392 0.062387 0.062360 0.062353 0.062346 0.062338 0.062336
  [353] 0.062319 0.062315 0.062293 0.062287 0.062253 0.062251 0.062232 0.062232
  [361] 0.062223 0.062190 0.062188 0.062165 0.062154 0.062139 0.062132 0.062119
  [369] 0.062119 0.062092 0.062084 0.062054 0.062048 0.062047 0.062044 0.062035
  [377] 0.062035 0.061998 0.061986 0.061972 0.061970 0.061961 0.061961 0.061936
  [385] 0.061915 0.061914 0.061906 0.061898 0.061876 0.061844 0.061824 0.061801
  [393] 0.061788 0.061786 0.061781 0.061770 0.061758 0.061754 0.061754 0.061744
  [4

In [51]:
median(rels_all$theta)
median(within_bay$theta)

[1] 0.0126005

[1] 0.054613

In [54]:
mayberels<-rels_all[rels_all$theta>0.25,]

In [55]:
mayberels[order(-mayberels$theta),c('IndA','IndB','theta')]

,IndA,IndB,theta
,<chr>,<chr>,<dbl>
26786,ACR_502,ACR_505,0.411440
25961,ACR_416,ACR_419,0.403595


In [66]:
bamlist<-function(ind){
    a<-paste0('/scratch/jbos/Moz_aligned_mil/amillepora_bamfiles/',ind)
    b<-paste0(a,'.sorted.bam')
    return(b)
    }

In [62]:
which(indlist=='ACR_505')
which(indlist=='ACR_419')

[1] 195

[1] 178

In [64]:
indlist_noclones<-indlist[-c(195,178)]
length(indlist_noclones)

[1] 234

In [67]:
a<-indlist_noclones
a<-bamlist(a)
write.table(a,'/home/jbos/Moz_reads/bam_names_grp137_noclones.txt',row.names=FALSE,col.names=FALSE,quote=FALSE)

In [68]:
bamlist7<-read.table('/home/jbos/Moz_reads/bam_names_grp7.txt')

In [69]:
indlist7<-apply(bamlist7,FUN=bamnames,MARGIN=1)

In [70]:
which(indlist7=='ACR_505')
which(indlist7=='ACR_419')

[1] 36

[1] 19

In [71]:
indlist7_noclones<-indlist7[-c(19,36)]

In [72]:
a<-indlist7_noclones
a<-bamlist(a)
write.table(a,'/home/jbos/Moz_reads/bam_names_grp7_noclones.txt',row.names=FALSE,col.names=FALSE,quote=FALSE)

In [74]:
bamlist37<-read.table('/home/jbos/Moz_reads/bam_names_grp37.txt')

In [75]:
indlist37<-apply(bamlist37,FUN=bamnames,MARGIN=1)

In [76]:
which(indlist37=='ACR_505')
which(indlist37=='ACR_419')

[1] 122

[1] 105

In [77]:
indlist37_noclones<-indlist37[-c(105,122)]

In [78]:
a<-indlist37_noclones
a<-bamlist(a)
write.table(a,'/home/jbos/Moz_reads/bam_names_grp37_noclones.txt',row.names=FALSE,col.names=FALSE,quote=FALSE)